# MODEL Monitoring Pipeline

In [1]:
import subprocess
import sys
import os

def get_repo_root():
    return subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode('utf-8').strip()
repo_root = get_repo_root()
print(repo_root)

src_path = os.path.join(repo_root, 'src')
# Add src_path to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

/home/sagemaker-user/aai-540-su25-group4


In [4]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
from datetime import datetime

role = get_execution_role()

region = boto3.Session().region_name

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")

sess = sagemaker.Session()

# project bucket
bucket_name = "aai-540-data"

# Configure s3 locations and sagemaker model to use
s3_production_meta_uri = f"s3://{bucket_name}/dev_split/production-meta.csv"
s3_label_map_uri = f"s3://{bucket_name}/dev_split/label_mapping.json"

In [5]:
# Load Model Package arn from notebook 03.
from sagemaker import ModelPackage
# 1. Create Model resource from package ARN
model = ModelPackage(
    model_package_arn='arn:aws:sagemaker:us-east-1:324183265896:model-package/wildscan-image-classifiers/1',
    role=role,
    sagemaker_session=sess
)
model.create()

-----
## Generate Monthly Ground Truth Data from Production Set

In [6]:
# try importing src/utils
from utils.utils import parse_s3_uri
from utils.utils import generate_manifest_file

In [7]:
# Process monthly data
def upload_monthly_ground_truth(df, year_month):
    year_str, month_str = year_month.split('-')
    year = int(year_str)
    month = int(month_str)
    
    start_date = f'{year}-{month:02d}-01'
    end_date = f'{year}-{month+1:02d}-01' if month < 12 else f'{year+1}-01-01'
    
    monthly_data = df[(df['timestamp'] >= start_date) & (df['timestamp'] < end_date)]
    monthly_data.to_csv(f'monthly_{year}_{month}.csv', index=False)
    
    s3 = boto3.client('s3')
    s3.upload_file(
        f'monthly_{year}_{month}.csv',
        bucket_name,
        f'ground_truth/{year}/{month}/labels.csv'
    )
    return f's3://{bucket_name}/ground_truth/{year}/{month}/labels.csv'


In [8]:
# access production-meta.csv from s3 bucket 

df = pd.read_csv(s3_production_meta_uri)
df['timestamp'] = pd.to_datetime(df['date_captured'])

# Generate list of year-months where there were image samples present
sample_year_months = df['timestamp'].dt.to_period('M').unique()
sample_year_months_str = sample_year_months.astype(str).tolist()

# Split the Production Set csv into Months and save into ground_truth/year/month folders
# generate a manifest file also for each month csv for easier batch transform. This should be added to a processing pipeline later
for year_month in sample_year_months_str:
    s3_uri = upload_monthly_ground_truth(df, year_month)
    print(s3_uri)
    generate_manifest_file(s3_input_csv = s3_uri, s3_images_loc = f"s3://{bucket_name}/cct_resized/")
    

s3://aai-540-data/ground_truth/2012/5/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/5/labels.manifest
s3://aai-540-data/ground_truth/2012/6/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/6/labels.manifest
s3://aai-540-data/ground_truth/2012/7/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/7/labels.manifest
s3://aai-540-data/ground_truth/2012/8/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/8/labels.manifest
s3://aai-540-data/ground_truth/2012/9/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/9/labels.manifest
s3://aai-540-data/ground_truth/2012/10/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/10/labels.manifest
s3://aai-540-data/ground_truth/2012/11/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/11/labels.manifest
s3://aai-540-data/ground_truth/2012/12/labels.csv
File uploaded to s3://aai-540-data/ground_truth/2012/12/labels.manifest
s3://aai-540-data/ground_truth/201

----
## Design Monitoring Pipeline

In [15]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.steps import TransformStep, ProcessingStep
from sagemaker.transformer import Transformer
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.functions import Join
from sagemaker.inputs import TransformInput

### Pipeline Input Parameters

In [11]:
# Pipeline parameters

# s3 uri folder where a csv and manifest file of new images collected in that month are present, with annotations, this is the ground truth folder
# and this is where evaluation files are going to be stored
year_month = ParameterString(name="Year_Month", default_value = "2012-05")
monthly_s3_uri = ParameterString(name="MonthlyS3Uri", default_value=f"s3://{bucket_name}/ground_truth/12/5")
model_name_param = ParameterString(name="model_name", default_value = model.name)
pipeline_timestamp = ParameterString(name="PipelineTimestamp",default_value="") # to simulate production set time-series performance on cloudwatch

### Configure Transformer Step 

In [13]:
# set transformer output s3 location
s3_transform_out = Join( on='/', values=[monthly_s3_uri, "batch_transform_out"])

# set manifest file s3 loc
s3_manifest_file = Join( on='/', values=[monthly_s3_uri, "labels.manifest"])

# initialize Tranformer
transformer = Transformer(
    model_name = model_name_param,
    instance_count=1,  # Number of instances
    instance_type="ml.g4dn.xlarge",  # Instance type
    output_path= s3_transform_out,  # Predictions output
    max_payload=10,  # Max payload size (MB)
    strategy="MultiRecord" , # for faster processing, but in real world, instance type can be ml.m5.xlarge and single record strategy is ok
    max_concurrent_transforms=10,
    sagemaker_session=sess,

    accept = 'txt/csv', # so output is generated in single file
    assemble_with='Line', # new line is generated for each prediction

)

# configure transformer STep
transform_step = TransformStep(
    name= 'MonthlyBatchTransform',
    transformer = transformer,
    inputs = TransformInput(
                data=s3_manifest_file,
                data_type='ManifestFile', # provide list of s3uris of objects to be batch transformed
                content_type='application/x-image', 
                split_type='None'
            )
)

### Configure Evaluation Step

In [16]:
# retrieve image_uri for evaluation script processor container
image_uri = sagemaker.image_uris.retrieve(
    framework='sklearn',        # or 'xgboost', 'pytorch', etc.
    region=region,
    version='1.2-1',            # Specify the version you need
    py_version='py3',           # Specify Python version if required
       # Use 'processing' for processing jobs
)

# Define your processing container (can use a built-in or custom image)
evaluation_processor = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,  # e.g., a scikit-learn or custom image
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    
)

s3_evaluation_out = Join( on='/', values=[monthly_s3_uri, "evaluation"])
s3_true_meta_uri = Join( on='/', values=[monthly_s3_uri, "labels.csv"])

# Define the evaluation Processing step
evaluation_step = ProcessingStep(
    name="ModelEvaluation",
    processor=evaluation_processor,
    code='../src/evaluation/evaluate.py',  # Your processing script,
    
    inputs=[
        # S3 location of batch transform predictions files
        ProcessingInput(
            source=transform_step.properties.TransformOutput.S3OutputPath,       # S3 bucket with predictions
            destination='/opt/ml/processing/input_predictions'        # Where the script will read input in local container
        ),
        
        # S3 location of the ground truth labels for the images in this set
        ProcessingInput(
            source=s3_true_meta_uri,
            destination='/opt/ml/processing/true_labels'
        ),

        # Label Mapping
        ProcessingInput(
            source=s3_label_map_uri,
            destination='/opt/ml/processing/label_mapping'
        )
    ],
    
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',           # Where the script will write output files in local container
            destination=s3_evaluation_out    # S3 bucket to store results
        )
    ]
)


INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


-----
## Assemble the Pipeline 
### (add more steps later like Conditional for Continuous D)

In [17]:
pipeline = Pipeline(
    name="MonthlyModelMonitoring",
    parameters=[year_month, monthly_s3_uri, model_name_param, pipeline_timestamp],
    steps=[transform_step, evaluation_step],
    sagemaker_session=sess
)

pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:324183265896:pipeline/MonthlyModelMonitoring',
 'ResponseMetadata': {'RequestId': 'b4dc19e0-9a20-4903-8b71-5f06216c9389',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b4dc19e0-9a20-4903-8b71-5f06216c9389',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '90',
   'date': 'Mon, 23 Jun 2025 21:06:34 GMT'},
  'RetryAttempts': 0}}

-----
## Execute Monitoring Pipeline

In [ ]:

for year_month in sample_year_months_str:
    year_str, month_str = year_month.split('-')
    year = int(year_str)
    month = int(month_str)

    # generate timestamp at the start of each pipeline execution
    # this is a workaround simulate the production set time-series nature and view the results in Cloud Watch
    
    # Generate the timestamp at pipeline start (e.g., end of month) -> this will be tied to the 'year_month' value in this iteration
    timestamp_value = datetime.utcnow().isoformat() + "Z"
    
    execution =pipeline.start(parameters={
        "Year_Month": year_month,
        "MonthlyS3Uri": f"s3://{bucket_name}/ground_truth/{year}/{month}",
        "model_name": model.name,
        "PipelineTimestamp": timestamp_value
    })

    execution.describe()
    execution.wait()


/tmp/ipykernel_8742/2570839740.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp_value = datetime.utcnow().isoformat() + "Z"


In [20]:
sample_year_months_str

['2012-05',
 '2012-06',
 '2012-07',
 '2012-08',
 '2012-09',
 '2012-10',
 '2012-11',
 '2012-12',
 '2013-01',
 '2013-02',
 '2013-03',
 '2013-04',
 '2013-05',
 '2013-08',
 '2013-09',
 '2013-10',
 '2013-11',
 '2013-12',
 '2014-01',
 '2014-02',
 '2014-03',
 '2014-04',
 '2014-05',
 '2014-06',
 '2014-07',
 '2014-08',
 '2014-09',
 '2014-10',
 '2014-11',
 '2014-12',
 '2015-01',
 '2015-02',
 '2015-03',
 '2015-04',
 '2015-05']